# Day 05 - Nulls, Duplicates, and Data Cleansing

**Topic:** Nulls Duplicates Cleansing  
**Main environment:** Microsoft Fabric Notebook + Fabric Lakehouse  
**Focus:** จัดการ dirty data พื้นฐาน เช่น null, blank string, duplicate records, duplicate business keys และสร้าง cleansing output ที่พร้อมใช้ต่อ

วันนี้เราจะฝึก mindset สำคัญของ Silver cleansing: ข้อมูล raw มักไม่สะอาดตั้งแต่ต้น เราจึงต้องตรวจ null, standardize ค่า, แยก invalid records ที่ critical, และ deduplicate ให้เหลือ record ที่เชื่อถือได้ก่อนเอาไปใช้ต่อ

## Goal of the day

1. ตรวจและนับ **null values** / **blank strings** ใน DataFrame ได้
2. ใช้ `fillna`, `dropna`, `isNull`, `isNotNull` ได้อย่างเหมาะสม
3. เข้าใจความต่างระหว่าง **exact duplicate** และ **business-key duplicate**
4. ใช้ `dropDuplicates()` และ deduplicate ด้วย business key + timestamp ได้
5. เพิ่ม **data quality flag columns** เพื่อบอกว่า record มี issue อะไรบ้าง
6. เขียน cleaned result เป็น Fabric Lakehouse table ด้วย `saveAsTable()` ได้

## Why it matters in production

ใน production pipeline ข้อมูล raw มักเจอปัญหาแบบนี้บ่อยมาก:

- required field เช่น `customer_id` เป็น null
- string field เป็นค่าว่าง `""` หรือมีช่องว่างเกินมา
- source ส่ง record ซ้ำมาแบบ exact duplicate
- business key เดิมถูกส่งมาหลาย version เช่น customer เดิมมีข้อมูล update ใหม่
- ถ้า drop records ทิ้งเงียบ ๆ จะ debug ยากและ audit ไม่ได้
- ถ้า deduplicate ผิด logic อาจเก็บ record เก่าแทน record ล่าสุด

Production takeaway:

> Cleansing ไม่ใช่แค่ลบ row ที่ดูสกปรก แต่ต้องทำให้ decision ชัดเจนว่าอะไรควร fill, อะไรควร flag, อะไรควร reject, และอะไรควร deduplicate ด้วย business rule

## Key concepts

| Concept | Meaning |
|---|---|
| `isNull()` | เช็คว่า column เป็น null |
| `isNotNull()` | เช็คว่า column ไม่เป็น null |
| Blank string | ค่า string ที่ไม่ใช่ null แต่เป็น `""` หรือช่องว่าง เช่น `"   "` |
| `fillna()` | เติม default value ให้ null values |
| `dropna()` | drop row ที่มี null ตามเงื่อนไขที่กำหนด |
| `dropDuplicates()` | ลบ duplicate rows หรือ duplicate ตาม subset columns |
| Exact duplicate | row ซ้ำแบบค่าทุก column เหมือนกัน |
| Business-key duplicate | key เดียวกันมีหลาย records เช่น `customer_id` เดียวกันแต่คนละ `updated_at` |
| Data quality flag | column ที่บอกว่า record มีปัญหาอะไร เช่น missing email, missing phone |
| Critical reject | record ที่ field สำคัญหาย เช่น ไม่มี business key หรือ timestamp ใช้งานไม่ได้ |

## Concept explanation

### Null vs blank string

ใน Spark ค่า `null` กับ blank string ไม่ใช่สิ่งเดียวกัน

```text
null      = ไม่มีค่า
""        = มีค่าเป็น string ว่าง
"   "     = มีค่าเป็นช่องว่าง ต้อง trim ก่อนถึงจะเห็นว่า blank
```

ดังนั้นการใช้แค่ `isNull()` อาจไม่พอสำหรับ string fields เช่น `email`, `phone`, `customer_name`

### Fill, drop, flag ใช้ต่างกันอย่างไร?

| Strategy | ใช้เมื่อไร |
|---|---|
| Fill | ค่า missing สามารถแทนด้วย default ได้โดยไม่ทำให้ business meaning เสีย เช่น `status = "unknown"` |
| Drop / Reject | field สำคัญมากจน record ใช้ต่อไม่ได้ เช่น `customer_id` หาย |
| Flag | record ยังใช้ต่อได้ แต่ต้องบอกว่ามี issue เช่น missing phone |

### Exact duplicate vs business-key duplicate

Exact duplicate คือ row ซ้ำเหมือนกันทั้ง row เช่น source ส่ง file เดิมซ้ำมา

Business-key duplicate คือ key เดียวกันมีหลาย version เช่น customer เดิมถูก update หลายครั้ง เราต้องเลือก record ที่ควร keep เช่น latest `updated_at`

ใน production ห้ามใช้ `dropDuplicates(["customer_id"])` แบบไม่คิด เพราะ Spark ไม่ guarantee ว่าจะเก็บ row ไหน ถ้าไม่มี ordering logic ที่ชัดเจน


## Mermaid diagram: Cleansing flow

```mermaid
flowchart LR
    A[Raw customer records] --> B[Standardize strings and timestamps]
    B --> C[Profile nulls and blanks]
    C --> D[Add data quality flags]
    D --> E{Critical issue?}
    E -- Yes --> R[Reject / review records]
    E -- No --> F[Remove exact duplicates]
    F --> G[Deduplicate by business key]
    G --> H[Cleaned Lakehouse table]
```

Key idea:

> Cleansing flow ควรทำให้เห็นทั้งข้อมูลที่ clean แล้ว และข้อมูลที่ถูก reject/flag เพื่อให้ตรวจสอบย้อนหลังได้

## Setup / imports

In [2]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

StatementMeta(, cf16f996-3c76-459e-9aa7-9ddde1b5da0f, 4, Finished, Available, Finished, False)

## Create mock data

Dataset นี้จำลอง customer records จากหลาย source ที่มี dirty data ปนอยู่ เช่น null, blank string, duplicate row, และ customer เดิมหลาย version

In [3]:
customers_data = [
    ("C001", "Alice", "ALICE@example.com", "081-111-2222", "active", "crm", "2026-05-01 09:00:00", "customer_20260501.csv"),
    ("C001", "Alice A.", "alice@example.com", "0811112222", "ACTIVE", "crm", "2026-05-03 10:30:00", "customer_20260503.csv"),
    ("C002", None, "bob@example.com", None, "active", "crm", "2026-05-01 09:05:00", "customer_20260501.csv"),
    ("C003", "Charlie", None, "082-333-4444", "inactive", "branch", "2026-05-01 10:00:00", "customer_20260501.csv"),
    (None, "No Id Customer", "noid@example.com", "083-555-6666", "active", "crm", "2026-05-01 11:00:00", "customer_20260501.csv"),
    ("C004", "  Daisy  ", "daisy@example.com", "   ", None, "branch", "2026-05-02 08:00:00", "customer_20260502.csv"),
    ("C005", "Eve", "eve@example.com", "084-777-8888", "active", "crm", "2026-05-02 12:00:00", "customer_20260502.csv"),
    ("C005", "Eve", "eve@example.com", "084-777-8888", "active", "crm", "2026-05-02 12:00:00", "customer_20260502.csv"),
    ("C006", "Frank", "FRANK@example.com", "085-999-0000", "active ", "crm", "2026-05-02 13:00:00", "customer_20260502.csv"),
    ("C006", "Frank", "frank@example.com", "0859990000", "inactive", "crm", "2026-05-04 15:00:00", "customer_20260504.csv"),
    ("C007", "Grace", "grace@example.com", "086-123-4567", "active", "crm", "invalid_timestamp", "customer_bad.csv"),
]

customer_schema = T.StructType([
    T.StructField("customer_id", T.StringType(), True),
    T.StructField("customer_name", T.StringType(), True),
    T.StructField("email", T.StringType(), True),
    T.StructField("phone", T.StringType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("source_system", T.StringType(), True),
    T.StructField("updated_at", T.StringType(), True),
    T.StructField("source_file", T.StringType(), True),
])

df_customers_raw = spark.createDataFrame(customers_data, customer_schema)

df_customers_raw.show(truncate=False)
df_customers_raw.printSchema()
print("Raw customer count:", df_customers_raw.count())

StatementMeta(, cf16f996-3c76-459e-9aa7-9ddde1b5da0f, 5, Finished, Available, Finished, False)

+-----------+--------------+-----------------+------------+--------+-------------+-------------------+---------------------+
|customer_id|customer_name |email            |phone       |status  |source_system|updated_at         |source_file          |
+-----------+--------------+-----------------+------------+--------+-------------+-------------------+---------------------+
|C001       |Alice         |ALICE@example.com|081-111-2222|active  |crm          |2026-05-01 09:00:00|customer_20260501.csv|
|C001       |Alice A.      |alice@example.com|0811112222  |ACTIVE  |crm          |2026-05-03 10:30:00|customer_20260503.csv|
|C002       |NULL          |bob@example.com  |NULL        |active  |crm          |2026-05-01 09:05:00|customer_20260501.csv|
|C003       |Charlie       |NULL             |082-333-4444|inactive|branch       |2026-05-01 10:00:00|customer_20260501.csv|
|NULL       |No Id Customer|noid@example.com |083-555-6666|active  |crm          |2026-05-01 11:00:00|customer_20260501.csv|


## PySpark code examples

ใน section นี้จะไล่จาก profiling → standardization → null handling → duplicate handling → quality flags → write Lakehouse table

### Example 1: Count nulls by column

Null profiling เป็น step แรก ๆ ที่ควรทำก่อน cleansing เพราะช่วยให้รู้ว่าคอลัมน์ไหนมี missing value มากน้อยแค่ไหน

In [5]:
null_count_exprs = [
    F.sum(F.when(F.col(column_name).isNull(), 1).otherwise(0)).alias(f"{column_name}_null_count")
    for column_name in df_customers_raw.columns
]

df_null_profile = df_customers_raw.agg(*null_count_exprs)  # * unpacks list into multiple agg expressions

df_null_profile.show(truncate=False)

StatementMeta(, cf16f996-3c76-459e-9aa7-9ddde1b5da0f, 7, Finished, Available, Finished, False)

+----------------------+------------------------+----------------+----------------+-----------------+------------------------+---------------------+----------------------+
|customer_id_null_count|customer_name_null_count|email_null_count|phone_null_count|status_null_count|source_system_null_count|updated_at_null_count|source_file_null_count|
+----------------------+------------------------+----------------+----------------+-----------------+------------------------+---------------------+----------------------+
|1                     |1                       |1               |1               |1                |0                       |0                    |0                     |
+----------------------+------------------------+----------------+----------------+-----------------+------------------------+---------------------+----------------------+



### Example 2: Count blank strings

สำหรับ string columns บางครั้งค่าไม่ได้เป็น null แต่เป็น `""` หรือช่องว่าง เช่น `"   "`  
ดังนั้นควร `trim()` ก่อนเช็ค blank

In [ ]:
string_columns = [
    field.name
    for field in df_customers_raw.schema.fields
    if isinstance(field.dataType, T.StringType)
]

# Tips:
# - string_columns keeps only text columns from the DataFrame schema
# - isinstance checks whether field.dataType is StringType
# - We filter StringType columns before using trim() because trim works on text

blank_count_exprs = [
    F.sum(
        F.when(
            F.col(column_name).isNotNull() & (F.trim(F.col(column_name)) == ""),
            1
        ).otherwise(0)
    ).alias(f"{column_name}_blank_count")
    for column_name in string_columns
]

df_blank_profile = df_customers_raw.agg(*blank_count_exprs)

df_blank_profile.show(truncate=False)

StatementMeta(, cf16f996-3c76-459e-9aa7-9ddde1b5da0f, 8, Finished, Available, Finished, False)

+-----------------------+-------------------------+-----------------+-----------------+------------------+-------------------------+----------------------+-----------------------+
|customer_id_blank_count|customer_name_blank_count|email_blank_count|phone_blank_count|status_blank_count|source_system_blank_count|updated_at_blank_count|source_file_blank_count|
+-----------------------+-------------------------+-----------------+-----------------+------------------+-------------------------+----------------------+-----------------------+
|0                      |0                        |0                |1                |0                 |0                        |0                     |0                      |
+-----------------------+-------------------------+-----------------+-----------------+------------------+-------------------------+----------------------+-----------------------+



### Example 3: Standardize raw values before cleansing

ก่อน deduplicate หรือ fill default values ควร standardize ค่าพื้นฐานก่อน เช่น:

- `trim` customer name
- `lower` email
- normalize status เป็น lowercase
- remove special characters from phone number
- parse `updated_at` เป็น timestamp

ถ้าไม่ standardize ก่อน อาจมองไม่เห็น duplicate เช่น `ALICE@example.com` กับ `alice@example.com`

In [ ]:
def is_null_or_blank(column_name):
    return F.col(column_name).isNull() | (
        F.trim(F.col(column_name).cast("string")) == ""  # Cast to string because trim works on text
    )


df_customers_standardized = (
    df_customers_raw
    .withColumn(
        "customer_id",
        F.when(is_null_or_blank("customer_id"), F.lit(None))
         .otherwise(F.upper(F.trim(F.col("customer_id"))))
    )
    .withColumn(
        "customer_name",
        F.when(is_null_or_blank("customer_name"), F.lit(None))
         .otherwise(F.trim(F.col("customer_name")))
    )
    .withColumn(
        "email_normalized",
        F.when(is_null_or_blank("email"), F.lit(None))
         .otherwise(F.lower(F.trim(F.col("email"))))
    )
    .withColumn(
        "phone_digits",
        F.regexp_replace(F.coalesce(F.col("phone"), F.lit("")), "[^0-9]", "")
    )
    .withColumn(
        "phone_digits",
        F.when(F.col("phone_digits") == "", F.lit(None)).otherwise(F.col("phone_digits"))  # Convert blank phone to null
    )
    .withColumn(
        "status_normalized",
        F.when(is_null_or_blank("status"), F.lit(None))
         .otherwise(F.lower(F.trim(F.col("status"))))
    )
    .withColumn("updated_at_ts", F.to_timestamp(F.col("updated_at"), "yyyy-MM-dd HH:mm:ss"))
)

# Tips:
# - coalesce checks each value in order and returns the first non-null value
# - regexp_replace(input_column, regex_pattern, replacement_value)
# - [^0-9] matches non-digit characters

df_customers_standardized.select(
    "customer_id",
    "customer_name",
    "email",
    "email_normalized",
    "phone",
    "phone_digits",
    "status",
    "status_normalized",
    "updated_at",
    "updated_at_ts",
    "source_file"
).show(truncate=False)

StatementMeta(, cf16f996-3c76-459e-9aa7-9ddde1b5da0f, 9, Finished, Available, Finished, False)

+-----------+--------------+-----------------+-----------------+------------+------------+--------+-----------------+-------------------+-------------------+---------------------+
|customer_id|customer_name |email            |email_normalized |phone       |phone_digits|status  |status_normalized|updated_at         |updated_at_ts      |source_file          |
+-----------+--------------+-----------------+-----------------+------------+------------+--------+-----------------+-------------------+-------------------+---------------------+
|C001       |Alice         |ALICE@example.com|alice@example.com|081-111-2222|0811112222  |active  |active           |2026-05-01 09:00:00|2026-05-01 09:00:00|customer_20260501.csv|
|C001       |Alice A.      |alice@example.com|alice@example.com|0811112222  |0811112222  |ACTIVE  |active           |2026-05-03 10:30:00|2026-05-03 10:30:00|customer_20260503.csv|
|C002       |NULL          |bob@example.com  |bob@example.com  |NULL        |NULL        |active  |a

### Example 4: Fill nullable fields with safe default values

`fillna()` เหมาะกับ column ที่ default value ไม่ทำให้ business meaning เสีย เช่น `status = "unknown"`

แต่ไม่ควร fill key สำคัญอย่าง `customer_id` แบบมั่ว ๆ เพราะจะทำให้ key ปลอมปนเข้า dataset

In [8]:
df_customers_filled = (
    df_customers_standardized
    .fillna({
        "customer_name": "UNKNOWN_CUSTOMER",
        "status_normalized": "unknown",
    })
)

df_customers_filled.select(
    "customer_id",
    "customer_name",
    "email_normalized",
    "phone_digits",
    "status_normalized",
    "updated_at_ts"
).show(truncate=False)

StatementMeta(, cf16f996-3c76-459e-9aa7-9ddde1b5da0f, 10, Finished, Available, Finished, False)

+-----------+----------------+-----------------+------------+-----------------+-------------------+
|customer_id|customer_name   |email_normalized |phone_digits|status_normalized|updated_at_ts      |
+-----------+----------------+-----------------+------------+-----------------+-------------------+
|C001       |Alice           |alice@example.com|0811112222  |active           |2026-05-01 09:00:00|
|C001       |Alice A.        |alice@example.com|0811112222  |active           |2026-05-03 10:30:00|
|C002       |UNKNOWN_CUSTOMER|bob@example.com  |NULL        |active           |2026-05-01 09:05:00|
|C003       |Charlie         |NULL             |0823334444  |inactive         |2026-05-01 10:00:00|
|NULL       |No Id Customer  |noid@example.com |0835556666  |active           |2026-05-01 11:00:00|
|C004       |Daisy           |daisy@example.com|NULL        |unknown          |2026-05-02 08:00:00|
|C005       |Eve             |eve@example.com  |0847778888  |active           |2026-05-02 12:00:00|


### Example 5: Drop or reject critical invalid records

ในตัวอย่างนี้เราจะถือว่า record ใช้ต่อไม่ได้ถ้า:

- `customer_id` เป็น null
- `updated_at_ts` parse ไม่ได้

เราจะยังไม่ drop เงียบ ๆ แต่แยกเป็น `df_customer_rejects` เพื่อให้ตรวจสอบย้อนหลังได้

In [ ]:
df_customer_rejects = (
    df_customers_filled
    .filter(F.col("customer_id").isNull() | F.col("updated_at_ts").isNull())
    .withColumn(
        "reject_reason",
        F.concat_ws(
            ", ",
            F.when(F.col("customer_id").isNull(), F.lit("missing_customer_id")),
            F.when(F.col("updated_at_ts").isNull(), F.lit("invalid_updated_at"))
        )
    )
)

# Tips:
# - concat_ws means concat with separator
# - It joins values using a separator and skips nulls
#   - Example: F.concat_ws(" | ", F.lit("missing_email"), F.lit(None), F.lit("invalid_phone"))
#   - Result: missing_email | invalid_phone

df_customer_candidates = (
    df_customers_filled
    .filter(F.col("customer_id").isNotNull() & F.col("updated_at_ts").isNotNull())
)

print("Reject count:", df_customer_rejects.count())
print("Candidate count:", df_customer_candidates.count())

df_customer_rejects.select(
    "customer_id",
    "customer_name",
    "updated_at",
    "updated_at_ts",
    "reject_reason",
    "source_file"
).show(truncate=False)

StatementMeta(, cf16f996-3c76-459e-9aa7-9ddde1b5da0f, 11, Finished, Available, Finished, False)

Reject count: 2
Candidate count: 9
+-----------+--------------+-------------------+-------------------+-------------------+---------------------+
|customer_id|customer_name |updated_at         |updated_at_ts      |reject_reason      |source_file          |
+-----------+--------------+-------------------+-------------------+-------------------+---------------------+
|NULL       |No Id Customer|2026-05-01 11:00:00|2026-05-01 11:00:00|missing_customer_id|customer_20260501.csv|
|C007       |Grace         |invalid_timestamp  |NULL               |invalid_updated_at |customer_bad.csv     |
+-----------+--------------+-------------------+-------------------+-------------------+---------------------+



### Example 6: Remove exact duplicate rows

Exact duplicate คือ row ที่ซ้ำเหมือนกันทุก column หลัง standardization/fill แล้ว  
ใช้ `dropDuplicates()` แบบไม่ระบุ subset เพื่อลบ row ที่เหมือนกันทั้ง row

In [10]:
raw_candidate_count = df_customer_candidates.count()

df_no_exact_duplicates = df_customer_candidates.dropDuplicates()

no_exact_duplicate_count = df_no_exact_duplicates.count()

print("Candidate count before exact dedup:", raw_candidate_count)
print("Candidate count after exact dedup:", no_exact_duplicate_count)
print("Removed exact duplicate count:", raw_candidate_count - no_exact_duplicate_count)

StatementMeta(, cf16f996-3c76-459e-9aa7-9ddde1b5da0f, 12, Finished, Available, Finished, False)

Candidate count before exact dedup: 9
Candidate count after exact dedup: 8
Removed exact duplicate count: 1


### Example 7: Detect duplicate business keys

หลังจากลบ exact duplicates แล้ว ยังอาจมี duplicate ตาม business key เช่น `customer_id` เดียวกันมีหลาย version

In [11]:
df_duplicate_customer_keys = (
    df_no_exact_duplicates
    .groupBy("customer_id")
    .agg(F.count("*").alias("record_count"))
    .filter(F.col("record_count") > 1)
    .orderBy("customer_id")
)

df_duplicate_customer_keys.show(truncate=False)

StatementMeta(, cf16f996-3c76-459e-9aa7-9ddde1b5da0f, 13, Finished, Available, Finished, False)

+-----------+------------+
|customer_id|record_count|
+-----------+------------+
|C001       |2           |
|C006       |2           |
+-----------+------------+



### Example 8: Keep latest record per business key

สำหรับ duplicate business key เราจะ keep record ล่าสุดจาก `updated_at_ts`

> วันนี้ใช้ `Window` แบบเบื้องต้นเพื่อ solve deduplication problem ก่อน รายละเอียด window functions จะเรียนลึกขึ้นใน Day 07

In [ ]:
latest_customer_window = (
    Window
    .partitionBy("customer_id")
    .orderBy(F.col("updated_at_ts").desc(), F.col("source_file").desc())
)

df_latest_customers = (
    df_no_exact_duplicates
    .withColumn("row_number", F.row_number().over(latest_customer_window))
    .filter(F.col("row_number") == 1)
    .drop("row_number")
)

# Tips:
# - Window defines how rows are grouped and ordered for window functions
# - partitionBy("customer_id") restarts the calculation for each customer
# - orderBy(...desc()) puts the latest record first within each customer group
# - row_number().over(window) assigns 1, 2, 3... based on the window order
# - filter(row_number == 1) keeps only the latest record per customer

df_latest_customers.select(
    "customer_id",
    "customer_name",
    "email_normalized",
    "phone_digits",
    "status_normalized",
    "updated_at_ts",
    "source_file"
).orderBy("customer_id").show(truncate=False)

StatementMeta(, cf16f996-3c76-459e-9aa7-9ddde1b5da0f, 14, Finished, Available, Finished, False)

+-----------+----------------+-----------------+------------+-----------------+-------------------+---------------------+
|customer_id|customer_name   |email_normalized |phone_digits|status_normalized|updated_at_ts      |source_file          |
+-----------+----------------+-----------------+------------+-----------------+-------------------+---------------------+
|C001       |Alice A.        |alice@example.com|0811112222  |active           |2026-05-03 10:30:00|customer_20260503.csv|
|C002       |UNKNOWN_CUSTOMER|bob@example.com  |NULL        |active           |2026-05-01 09:05:00|customer_20260501.csv|
|C003       |Charlie         |NULL             |0823334444  |inactive         |2026-05-01 10:00:00|customer_20260501.csv|
|C004       |Daisy           |daisy@example.com|NULL        |unknown          |2026-05-02 08:00:00|customer_20260502.csv|
|C005       |Eve             |eve@example.com  |0847778888  |active           |2026-05-02 12:00:00|customer_20260502.csv|
|C006       |Frank      

### Example 9: Add data quality flags

บาง issue ไม่จำเป็นต้อง reject ทันที เช่น missing phone หรือ missing email  
แต่ควร flag ไว้ เพื่อให้ downstream หรือ data owner เห็นว่า record นี้มี warning

In [13]:
df_customers_cleaned = (
    df_latest_customers
    .withColumn(
        "quality_issue_reason",
        F.concat_ws(
            ", ",
            F.when(F.col("email_normalized").isNull(), F.lit("missing_email")),
            F.when(F.col("phone_digits").isNull(), F.lit("missing_phone")),
            F.when(~F.col("status_normalized").isin("active", "inactive", "unknown"), F.lit("invalid_status"))
        )
    )
    .withColumn("has_quality_issue", F.length(F.col("quality_issue_reason")) > 0)
    .select(
        "customer_id",
        "customer_name",
        "email_normalized",
        "phone_digits",
        "status_normalized",
        "source_system",
        "updated_at_ts",
        "source_file",
        "has_quality_issue",
        "quality_issue_reason"
    )
    .orderBy("customer_id")
)

df_customers_cleaned.show(truncate=False)

StatementMeta(, cf16f996-3c76-459e-9aa7-9ddde1b5da0f, 15, Finished, Available, Finished, False)

+-----------+----------------+-----------------+------------+-----------------+-------------+-------------------+---------------------+-----------------+--------------------+
|customer_id|customer_name   |email_normalized |phone_digits|status_normalized|source_system|updated_at_ts      |source_file          |has_quality_issue|quality_issue_reason|
+-----------+----------------+-----------------+------------+-----------------+-------------+-------------------+---------------------+-----------------+--------------------+
|C001       |Alice A.        |alice@example.com|0811112222  |active           |crm          |2026-05-03 10:30:00|customer_20260503.csv|false            |                    |
|C002       |UNKNOWN_CUSTOMER|bob@example.com  |NULL        |active           |crm          |2026-05-01 09:05:00|customer_20260501.csv|true             |missing_phone       |
|C003       |Charlie         |NULL             |0823334444  |inactive         |branch       |2026-05-01 10:00:00|customer_202

### Example 10: Validate cleaned dataset

หลัง cleansing ควรมี validation ขั้นพื้นฐาน เช่น:

- ไม่มี `customer_id` เป็น null
- ไม่มี duplicate `customer_id`
- จำนวน cleaned records สมเหตุสมผล

In [14]:
missing_key_count = df_customers_cleaned.filter(F.col("customer_id").isNull()).count()

duplicate_key_count = (
    df_customers_cleaned
    .groupBy("customer_id")
    .agg(F.count("*").alias("record_count"))
    .filter(F.col("record_count") > 1)
    .count()
)

print("Cleaned customer count:", df_customers_cleaned.count())
print("Missing customer_id count:", missing_key_count)
print("Duplicate customer_id count:", duplicate_key_count)

StatementMeta(, cf16f996-3c76-459e-9aa7-9ddde1b5da0f, 16, Finished, Available, Finished, False)

Cleaned customer count: 6
Missing customer_id count: 0
Duplicate customer_id count: 0


### Example 11: Write cleaned and reject records to Lakehouse tables

ใน Fabric Lakehouse, `saveAsTable()` จะเขียน DataFrame เป็น Lakehouse table

> หมายเหตุ: ต้อง attach Lakehouse เข้ากับ Fabric Notebook ก่อนรัน cell นี้

In [15]:
cleaned_table_name = "day05_customer_cleaned"
reject_table_name = "day05_customer_rejects"

(
    df_customers_cleaned
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(cleaned_table_name)
)

(
    df_customer_rejects
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(reject_table_name)
)

print(f"Table written successfully: {cleaned_table_name}")
print(f"Table written successfully: {reject_table_name}")

StatementMeta(, cf16f996-3c76-459e-9aa7-9ddde1b5da0f, 17, Finished, Available, Finished, False)

Table written successfully: day05_customer_cleaned
Table written successfully: day05_customer_rejects


### Example 12: Read Lakehouse tables back

การ read กลับช่วยเช็คว่า table ถูกสร้างแล้วจริง และ schema/records ยังถูกต้องหลัง write

In [16]:
spark.read.table("day05_customer_cleaned").show(truncate=False)
spark.read.table("day05_customer_rejects").show(truncate=False)

StatementMeta(, cf16f996-3c76-459e-9aa7-9ddde1b5da0f, 18, Finished, Available, Finished, False)

+-----------+----------------+-----------------+------------+-----------------+-------------+-------------------+---------------------+-----------------+--------------------+
|customer_id|customer_name   |email_normalized |phone_digits|status_normalized|source_system|updated_at_ts      |source_file          |has_quality_issue|quality_issue_reason|
+-----------+----------------+-----------------+------------+-----------------+-------------+-------------------+---------------------+-----------------+--------------------+
|C001       |Alice A.        |alice@example.com|0811112222  |active           |crm          |2026-05-03 10:30:00|customer_20260503.csv|false            |                    |
|C002       |UNKNOWN_CUSTOMER|bob@example.com  |NULL        |active           |crm          |2026-05-01 09:05:00|customer_20260501.csv|true             |missing_phone       |
|C003       |Charlie         |NULL             |0823334444  |inactive         |branch       |2026-05-01 10:00:00|customer_202

## Quick recap

| Question | Answer |
|---|---|
| Null กับ blank string เหมือนกันไหม? | ไม่เหมือนกัน `isNull()` จับ null แต่ blank ต้องใช้ `trim()` ตรวจเพิ่ม |
| `fillna()` เหมาะกับอะไร? | เติม default ให้ nullable field ที่ไม่ใช่ key สำคัญ |
| ควร drop invalid records เงียบ ๆ ไหม? | ไม่ควร อย่างน้อยควรแยก reject/flag เพื่อ debug ได้ |
| Exact duplicate คืออะไร? | row ซ้ำเหมือนกันทุก column |
| Business-key duplicate คืออะไร? | key เดียวกันมีหลาย version ต้องเลือก record ด้วย business rule |
| `dropDuplicates(["customer_id"])` เสี่ยงอะไร? | Spark อาจเก็บ row ไหนก็ได้ถ้าไม่มี ordering logic |
| วิธีเลือก latest record ใช้อะไรได้? | ใช้ `Window.partitionBy(...).orderBy(...)` + `row_number()` |

## Exercises

### Exercise 1: Build null and blank profile

ใช้ `df_customers_raw` เพื่อสร้าง profile 2 ชุด:

1. `df_ex_null_profile` นับจำนวน null ของทุก column
2. `df_ex_blank_profile` นับจำนวน blank string ของ string columns

Expected result:

- เห็นจำนวน null ใน field เช่น `customer_id`, `customer_name`, `email`, `phone`, `status`
- เห็น blank string ใน field เช่น `phone`

In [17]:
exercise_null_exprs = [
    F.sum(F.when(F.col(column_name).isNull(), 1).otherwise(0)).alias(f"{column_name}_null_count")
    for column_name in df_customers_raw.columns
]

df_ex_null_profile = df_customers_raw.agg(*exercise_null_exprs)  # * unpacks list into multiple agg expressions

df_ex_blank_profile = df_customers_raw.agg(*[
    F.sum(
        F.when(
            F.col(column_name).isNotNull() & (F.trim(F.col(column_name)) == ""),
            1
        ).otherwise(0)
    ).alias(f"{column_name}_blank_count")
    for column_name in string_columns
])

df_ex_null_profile.show(truncate=False)
df_ex_blank_profile.show(truncate=False)

StatementMeta(, cf16f996-3c76-459e-9aa7-9ddde1b5da0f, 19, Finished, Available, Finished, False)

+----------------------+------------------------+----------------+----------------+-----------------+------------------------+---------------------+----------------------+
|customer_id_null_count|customer_name_null_count|email_null_count|phone_null_count|status_null_count|source_system_null_count|updated_at_null_count|source_file_null_count|
+----------------------+------------------------+----------------+----------------+-----------------+------------------------+---------------------+----------------------+
|1                     |1                       |1               |1               |1                |0                       |0                    |0                     |
+----------------------+------------------------+----------------+----------------+-----------------+------------------------+---------------------+----------------------+

+-----------------------+-------------------------+-----------------+-----------------+------------------+-------------------------+-------

### Exercise 2: Create quality flags

สร้าง DataFrame ชื่อ `df_ex_quality_flags` จาก `df_customers_standardized`

Requirements:

- เพิ่ม `missing_customer_id_flag`
- เพิ่ม `missing_email_flag`
- เพิ่ม `missing_phone_flag`
- เพิ่ม `invalid_updated_at_flag`
- เพิ่ม `has_any_issue` เป็น `True` ถ้ามี issue อย่างน้อย 1 อย่าง

Expected result:

- records ที่ไม่มี `customer_id`, ไม่มี email, ไม่มี phone หรือ timestamp parse ไม่ได้ ควรถูก flag เป็น `True`

In [18]:
df_ex_quality_flags = (
    df_customers_standardized
    .withColumn("missing_customer_id_flag", F.col("customer_id").isNull())
    .withColumn("missing_email_flag", F.col("email_normalized").isNull())
    .withColumn("missing_phone_flag", F.col("phone_digits").isNull())
    .withColumn("invalid_updated_at_flag", F.col("updated_at_ts").isNull())
    .withColumn(
        "has_any_issue",
        F.col("missing_customer_id_flag") |
        F.col("missing_email_flag") |
        F.col("missing_phone_flag") |
        F.col("invalid_updated_at_flag")
    )
)

df_ex_quality_flags.select(
    "customer_id",
    "customer_name",
    "missing_customer_id_flag",
    "missing_email_flag",
    "missing_phone_flag",
    "invalid_updated_at_flag",
    "has_any_issue"
).show(truncate=False)

StatementMeta(, cf16f996-3c76-459e-9aa7-9ddde1b5da0f, 20, Finished, Available, Finished, False)

+-----------+--------------+------------------------+------------------+------------------+-----------------------+-------------+
|customer_id|customer_name |missing_customer_id_flag|missing_email_flag|missing_phone_flag|invalid_updated_at_flag|has_any_issue|
+-----------+--------------+------------------------+------------------+------------------+-----------------------+-------------+
|C001       |Alice         |false                   |false             |false             |false                  |false        |
|C001       |Alice A.      |false                   |false             |false             |false                  |false        |
|C002       |NULL          |false                   |false             |true              |false                  |true         |
|C003       |Charlie       |false                   |true              |false             |false                  |true         |
|NULL       |No Id Customer|true                    |false             |false             

### Exercise 3: Deduplicate and write final table

สร้าง DataFrame ชื่อ `df_ex_customer_latest_cleaned`

Requirements:

- filter records ที่มี `customer_id` และ `updated_at_ts` ใช้งานได้
- remove exact duplicates
- keep latest record ต่อ `customer_id`
- write เป็น Lakehouse table ชื่อ `day05_ex_customer_latest_cleaned`
- read table กลับมา `.show()`

Expected result:

- table มีหนึ่ง record ต่อหนึ่ง `customer_id`
- `C001` และ `C006` ควรเหลือ latest version

In [ ]:
exercise_latest_window = (
    Window
    .partitionBy("customer_id")
    .orderBy(F.col("updated_at_ts").desc(), F.col("source_file").desc())
)

df_ex_customer_latest_cleaned = (
    df_customers_standardized
    .filter(F.col("customer_id").isNotNull() & F.col("updated_at_ts").isNotNull())
    .withColumn("row_number", F.row_number().over(exercise_latest_window))
    .filter(F.col("row_number") == 1)
    .drop("row_number")
    .select(
        "customer_id",
        "customer_name",
        "email_normalized",
        "phone_digits",
        "status_normalized",
        "updated_at_ts",
        "source_file"
    )
    .orderBy("customer_id")
)

# Tips:
# - row_number + filter(row_number == 1) keeps the latest record per customer_id
# - dropDuplicates() is not needed here because the Window logic already returns one row per customer_id

(
    df_ex_customer_latest_cleaned
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("day05_ex_customer_latest_cleaned")
)

spark.read.table("day05_ex_customer_latest_cleaned").show(truncate=False)

StatementMeta(, cf16f996-3c76-459e-9aa7-9ddde1b5da0f, 26, Finished, Available, Finished, False)

+-----------+-------------+-----------------+------------+-----------------+-------------------+---------------------+
|customer_id|customer_name|email_normalized |phone_digits|status_normalized|updated_at_ts      |source_file          |
+-----------+-------------+-----------------+------------+-----------------+-------------------+---------------------+
|C001       |Alice A.     |alice@example.com|0811112222  |active           |2026-05-03 10:30:00|customer_20260503.csv|
|C002       |NULL         |bob@example.com  |NULL        |active           |2026-05-01 09:05:00|customer_20260501.csv|
|C003       |Charlie      |NULL             |0823334444  |inactive         |2026-05-01 10:00:00|customer_20260501.csv|
|C004       |Daisy        |daisy@example.com|NULL        |NULL             |2026-05-02 08:00:00|customer_20260502.csv|
|C005       |Eve          |eve@example.com  |0847778888  |active           |2026-05-02 12:00:00|customer_20260502.csv|
|C006       |Frank        |frank@example.com|085

## Common mistakes

1. **คิดว่า `fillna()` จัดการ blank string ได้ทั้งหมด**

   `fillna()` เติมเฉพาะ null แต่ไม่จัดการ `""` หรือ `"   "` ต้องใช้ `trim()` และ condition เพิ่ม

2. **ใช้ `dropna()` แบบกว้างเกินไป**

   เช่น drop ทุก row ที่มี null ใน column ใดก็ได้ อาจทำให้ข้อมูลหายเยอะเกินจำเป็น ควรกำหนด `subset` ตาม required fields

3. **Fill business key ด้วย default value**

   ไม่ควรเติม `customer_id = "UNKNOWN"` เพื่อให้ record ผ่าน เพราะจะสร้าง key ปลอมและทำให้ downstream join ผิด

4. **ใช้ `dropDuplicates(["customer_id"])` โดยไม่สนใจ latest record**

   ถ้า key เดียวกันมีหลาย version ต้องมี rule เช่น keep latest `updated_at` ไม่ใช่ปล่อยให้ Spark เลือก row เอง

5. **Deduplicate ก่อน standardize**

   ถ้าไม่ `trim`, `lower`, normalize phone/status ก่อน อาจมองไม่เห็น duplicate ที่จริง ๆ คือ entity เดียวกัน

6. **Drop bad records เงียบ ๆ**

   ใน production ควรมี reject/flag reason อย่างน้อย เพื่อให้ audit และ debug ได้ว่า record หายไปเพราะอะไร

## Expected Output / Success Criteria

เมื่อจบ Day 05 ควรทำได้:

- อธิบายได้ว่า `null` กับ blank string ต่างกันอย่างไร
- ใช้ `isNull()`, `isNotNull()`, `trim()` เพื่อ profile missing values ได้
- ใช้ `fillna()` กับ nullable fields ที่เหมาะสมได้
- อธิบายได้ว่า field สำคัญ เช่น business key ไม่ควรถูก fill แบบมั่ว ๆ
- แยก critical invalid records ออกจาก candidates ได้
- ใช้ `dropDuplicates()` เพื่อลบ exact duplicate rows ได้
- ตรวจ duplicate business key ด้วย `groupBy().count()` ได้
- ใช้ business rule เพื่อ keep latest record ต่อ key ได้
- สร้าง data quality flags เช่น `has_quality_issue`, `quality_issue_reason` ได้
- เขียน cleaned table และ reject table ลง Fabric Lakehouse ได้
- อ่าน table กลับด้วย `spark.read.table()` เพื่อตรวจผลลัพธ์ได้

## Final takeaway

Nulls, duplicates, และ dirty values เป็นเรื่องปกติของ raw data

> งานของ Data Engineer ไม่ใช่แค่ทำให้ error หาย แต่ต้องทำให้ cleansing decision โปร่งใส ตรวจสอบได้ และ rerun แล้วผลลัพธ์ยังน่าเชื่อถือ

สำหรับ Day 05 mindset ที่ควรจำคือ:

- Profile ก่อน clean
- Standardize ก่อน deduplicate
- Fill เฉพาะ field ที่เหมาะสม
- Reject/flag records ที่มีปัญหา ไม่ใช่ drop เงียบ ๆ
- Deduplicate ด้วย business rule ไม่ใช่แค่ลบ record ซ้ำแบบสุ่ม
- Cleaned table ควรมีหนึ่ง record ต่อ business key ตาม rule ที่อธิบายได้

## Optional cleanup

In [ ]:
# spark.sql("DROP TABLE IF EXISTS day05_customer_cleaned")
# spark.sql("DROP TABLE IF EXISTS day05_customer_rejects")
# spark.sql("DROP TABLE IF EXISTS day05_ex_customer_latest_cleaned")